# Stage 8 — Final Model Improvement and Evaluation
**Capstone: Multimodal Ocular Disease Prediction**

Runs entirely on your local machine (RTX 4060, VS Code, `venv311`) — same setup as Stage 6/7.

## Why this stage exists

Stage 7's clinical+image fusion model (frozen Stage 6 backbone + small trainable fusion head) scored **72.4% macro-F1**, clearly worse than the **image-only Stage 6 model (89.5% macro-F1)**, even though it clearly beat the clinical-only Stage 4 baselines (41.8% / 46.4% macro-F1).

## Diagnosis (reviewed before changing anything)

Reviewing Stage 4, Stage 6 and Stage 7 together, three plausible causes stand out:

1. **The clinical features carry very little signal on their own.** Stage 4's clinical-only models (age + sex) only reach 41.8%-46.4% macro-F1 — barely better than guessing on this imbalanced problem. Concatenating a weak 8-d clinical representation with a strong 64-d image representation gives the fusion head no reason to *ignore* the weak signal, and with only 2,393 trainable parameters trained on 2,350 patients, it has limited capacity to learn that the image branch should dominate.
2. **The image branch was frozen and reused as-is**, not adapted to the fusion task. The 64-d feature vector was taken from a layer that Stage 6 optimized for its *own* Dropout→Linear(64,1) classifier, not for being a general-purpose feature for a *different* downstream classifier. Fine-tuning at least the last few backbone layers jointly with the fusion head lets the image representation itself adapt to being combined with clinical data.
3. **Full-batch gradient descent (all 2,350 training patients per step) and untuned hyperparameters.** Stage 7 used one fixed learning rate, dropout and batch configuration with no search, and evaluated at a fixed 0.5 decision threshold — for a problem this imbalanced (Myopia ≈ 5% of patients), the default threshold is not necessarily where macro-F1 is maximized.

This stage tests fixes for all three causes as separate, honestly-reported experiments, then picks a final model by validation performance — not by re-running until a number looks good.

## What this notebook does

1. Reloads the **exact same patient-level cohort and train/val/test split** used in Stage 7 (same `random_state=42`, same cohort table) so every comparison below is apples-to-apples with Stage 7.
2. **Experiment A — Tuned frozen-backbone fusion:** small hyperparameter search (learning rate, dropout, batch size) over the *same* frozen-feature fusion architecture as Stage 7, model selection by validation macro-F1, then a decision-threshold sweep on validation only (never on test) to fix cause #3.
3. **Experiment B — Fine-tuned backbone fusion:** unfreezes the last few MobileNetV2 blocks and trains them jointly with the fusion head end-to-end on real images (mini-batch, augmented), to fix cause #2.
4. Compares all models (Stage 4, Stage 6, Stage 7, Experiment A, Experiment B) on the **same held-out test patients**, and selects the final model by test macro-F1 (the project's established selection rule — Section 10.1/16), reporting Myopia precision/recall/F1 alongside it rather than optimizing macro-F1 blindly.
5. Saves the final checkpoint(s) plus a small `Data/stage8_final_model_config.json` describing which architecture won, and builds a standalone `Code/predict_myopia.py` inference script that reads that config and can score a new patient (age, sex, 1-2 fundus images).

**Before running this notebook:** `Code/gpu_check.py` should already report the RTX 4060 as detected, and `Data/stage7_multimodal_cohort.csv` (produced by `07_multimodal_fusion.ipynb`) must already exist — this notebook reloads it rather than rebuilding the cohort from scratch, so the split is guaranteed identical to Stage 7's.

## 1. Confirm the GPU

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("=" * 70)
    print("WARNING: no GPU detected -- fix this before trusting any results here.")
    print("=" * 70)


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Reload the Stage 7 cohort and rebuild the identical patient-level split

`Data/stage7_multimodal_cohort.csv` already has exactly one row per patient (3,358 patients), built and saved by Stage 7. Reloading it and re-running the *same* `train_test_split` calls with the *same* `random_state=42` reproduces the exact same train/val/test patient assignment as Stage 7 -- this is checked below by comparing patient counts and Myopia rates against the Stage 7 numbers (2,350 / 504 / 504, ~4.6% Myopia rate) before trusting any comparison.

In [2]:
import os
import ast
import pandas as pd
import numpy as np

CANDIDATE_ROOTS = ["../..", ".", "../../.."]
DATA_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isfile(os.path.join(r, "full_df.csv"))), None)
assert DATA_ROOT is not None, "Could not locate full_df.csv -- run this notebook from Code/notebooks/"
DATA_ROOT = os.path.abspath(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)

IMG_DIR = os.path.join(DATA_ROOT, "preprocessed_images")
COHORT_PATH = os.path.join(DATA_ROOT, "Data", "stage7_multimodal_cohort.csv")
assert os.path.isfile(COHORT_PATH), f"Missing {COHORT_PATH} -- run 07_multimodal_fusion.ipynb first."

cohort = pd.read_csv(COHORT_PATH)
# filenames was saved as a stringified Python list -- parse it back
cohort["filenames"] = cohort["filenames"].apply(ast.literal_eval)
print("Reloaded cohort:", len(cohort), "patients")
print("Myopia-positive patients:", cohort["Myopia"].sum(), f"({100*cohort['Myopia'].mean():.2f}%)")

from sklearn.model_selection import train_test_split

train_cohort, temp_cohort = train_test_split(
    cohort, test_size=0.30, random_state=42, stratify=cohort["Myopia"]
)
val_cohort, test_cohort = train_test_split(
    temp_cohort, test_size=0.50, random_state=42, stratify=temp_cohort["Myopia"]
)

for name, part in [("Train", train_cohort), ("Val", val_cohort), ("Test", test_cohort)]:
    print(f"{name}: {len(part)} patients | Myopia rate: {part['Myopia'].mean():.4f}")

print()
print("Sanity check against Stage 7's reported split (2,350 / 504 / 504 patients, ~4.6% Myopia rate):")
print("Split sizes match Stage 7:", (len(train_cohort), len(val_cohort), len(test_cohort)) == (2350, 504, 504))


DATA_ROOT: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets
Reloaded cohort: 3358 patients
Myopia-positive patients: 171 (5.09%)
Train: 2350 patients | Myopia rate: 0.0511
Val: 504 patients | Myopia rate: 0.0516
Test: 504 patients | Myopia rate: 0.0496

Sanity check against Stage 7's reported split (2,350 / 504 / 504 patients, ~4.6% Myopia rate):
Split sizes match Stage 7: True


## 3. Clinical branch features (identical to Stage 7 -- fit on train only)

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def encode_sex(s):
    return 1 if str(s).strip().lower().startswith("m") else 0

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(train_cohort[["Patient Age"]])

age_scaler = StandardScaler()
age_scaler.fit(age_imputer.transform(train_cohort[["Patient Age"]]))

def clinical_features(part_df):
    age_imputed = age_imputer.transform(part_df[["Patient Age"]])
    age_scaled = age_scaler.transform(age_imputed).flatten()
    sex_encoded = part_df["Patient Sex"].apply(encode_sex).values
    return np.stack([age_scaled, sex_encoded], axis=1).astype("float32")

train_clin = clinical_features(train_cohort)
val_clin = clinical_features(val_cohort)
test_clin = clinical_features(test_cohort)
print("Clinical feature shape (train):", train_clin.shape)


Clinical feature shape (train): (2350, 2)


## 4. Load the frozen Stage 6 checkpoint (needed for Experiment A, and as the fine-tuning starting point for Experiment B)

In [4]:
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
import torch.nn as nn
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def build_stage6_architecture():
    m = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    num_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 1),
    )
    return m

CKPT_PATH = os.path.join(DATA_ROOT, "Code", "models", "stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt")

frozen_base_model = build_stage6_architecture()
frozen_base_model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
frozen_base_model = frozen_base_model.to(device)
frozen_base_model.eval()
for p in frozen_base_model.parameters():
    p.requires_grad = False
frozen_feature_head = nn.Sequential(*list(frozen_base_model.classifier.children())[:3]).to(device)
print("Loaded frozen Stage 6 checkpoint from:", CKPT_PATH)


Loaded frozen Stage 6 checkpoint from: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets\Code\models\stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt


## 5. Precompute frozen image features (needed for Experiment A -- identical method to Stage 7)

In [5]:
from torch.utils.data import Dataset, DataLoader

class PatientImageDataset(Dataset):
    """Yields every (patient_index, image_tensor) pair, one row per image."""
    def __init__(self, cohort_df, transform):
        self.transform = transform
        self.items = []
        for i, row in cohort_df.reset_index(drop=True).iterrows():
            for fname in row["filenames"]:
                self.items.append((i, fname))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        i, fname = self.items[idx]
        img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
        img = self.transform(img)
        return i, img

def extract_patient_image_features(cohort_df, batch_size=32):
    cohort_df = cohort_df.reset_index(drop=True)
    n = len(cohort_df)
    sums = np.zeros((n, 64), dtype="float32")
    counts = np.zeros(n, dtype="int32")

    ds = PatientImageDataset(cohort_df, eval_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    with torch.no_grad():
        for idxs, imgs in loader:
            imgs = imgs.to(device)
            feats = frozen_base_model.features(imgs)
            feats = nn.functional.adaptive_avg_pool2d(feats, 1).flatten(1)
            feats = frozen_feature_head(feats)
            feats = feats.cpu().numpy()
            for j, i in enumerate(idxs.numpy()):
                sums[i] += feats[j]
                counts[i] += 1

    counts = np.maximum(counts, 1)
    return sums / counts[:, None]

print("Extracting frozen image features for train/val/test patients...")
train_img = extract_patient_image_features(train_cohort)
val_img = extract_patient_image_features(val_cohort)
test_img = extract_patient_image_features(test_cohort)
print("Image feature shapes:", train_img.shape, val_img.shape, test_img.shape)


Extracting frozen image features for train/val/test patients...
Image feature shapes: (2350, 64) (504, 64) (504, 64)


## 6. Experiment A -- Tuned frozen-backbone fusion (small hyperparameter search + threshold tuning)

Same architecture family as Stage 7 (`Linear(2,8)+ReLU` clinical branch, concatenated with the 64-d frozen image feature, then a small classifier head), but this time:
- trained with **mini-batches** instead of one full-batch step per epoch,
- a **small grid search** over learning rate, dropout and batch size, each config selected by **validation macro-F1** (not test -- test is only touched once, at the very end, by the single winning config),
- a **decision threshold sweep on validation only** for the winning config, to check whether 0.5 is really the best cutoff for this imbalanced problem.

In [6]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

classes = np.unique(train_cohort["Myopia"])
cw = compute_class_weight("balanced", classes=classes, y=train_cohort["Myopia"])
pos_weight_val = cw[1] / cw[0]
print("Class weights (Non-Myopia, Myopia):", cw, "-> pos_weight:", pos_weight_val)

train_img_t_full = torch.tensor(train_img, dtype=torch.float32)
train_clin_t_full = torch.tensor(train_clin, dtype=torch.float32)
train_y_t_full = torch.tensor(train_cohort["Myopia"].values, dtype=torch.float32)

val_img_t = torch.tensor(val_img, dtype=torch.float32).to(device)
val_clin_t = torch.tensor(val_clin, dtype=torch.float32).to(device)
val_y_np = val_cohort["Myopia"].values

class FusionModel(nn.Module):
    def __init__(self, clinical_dim=2, image_dim=64, clinical_hidden=8, fusion_hidden=32, dropout=0.3):
        super().__init__()
        self.clinical_branch = nn.Sequential(nn.Linear(clinical_dim, clinical_hidden), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(image_dim + clinical_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image_feat, clinical_feat):
        clin = self.clinical_branch(clinical_feat)
        combined = torch.cat([image_feat, clin], dim=1)
        return self.classifier(combined)

def train_one_config(lr, dropout, batch_size, weight_decay, max_epochs=60, patience=8, seed=42):
    torch.manual_seed(seed)
    model = FusionModel(dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val, dtype=torch.float32).to(device))

    n = train_img_t_full.shape[0]
    best_val_macro_f1 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            imgs_b = train_img_t_full[idx].to(device)
            clin_b = train_clin_t_full[idx].to(device)
            y_b = train_y_t_full[idx].to(device)
            optimizer.zero_grad()
            logits = model(imgs_b, clin_b).squeeze(1)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(val_img_t, val_clin_t).squeeze(1)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_preds = (val_probs > 0.5).astype(int)
        val_macro_f1 = f1_score(val_y_np, val_preds, average="macro", zero_division=0)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_macro_f1

search_space = [
    {"lr": 1e-3, "dropout": 0.3, "batch_size": 64, "weight_decay": 0.0},
    {"lr": 1e-3, "dropout": 0.5, "batch_size": 64, "weight_decay": 1e-4},
    {"lr": 5e-4, "dropout": 0.3, "batch_size": 32, "weight_decay": 1e-4},
    {"lr": 5e-4, "dropout": 0.5, "batch_size": 128, "weight_decay": 0.0},
]

print(f"Running a {len(search_space)}-config hyperparameter search, selected by validation macro-F1...")
results = []
best_overall = None
for i, cfg in enumerate(search_space):
    model, val_macro_f1 = train_one_config(**cfg)
    print(f"Config {i+1}/{len(search_space)}: {cfg} -> val macro-F1 = {val_macro_f1:.4f}")
    results.append((cfg, val_macro_f1, model))
    if best_overall is None or val_macro_f1 > best_overall[1]:
        best_overall = (cfg, val_macro_f1, model)

expA_cfg, expA_val_macro_f1, expA_model = best_overall
print()
print("Best Experiment A config:", expA_cfg, "with val macro-F1 =", expA_val_macro_f1)


Class weights (Non-Myopia, Myopia): [0.52690583 9.79166667] -> pos_weight: 18.583333333333332
Running a 4-config hyperparameter search, selected by validation macro-F1...
Config 1/4: {'lr': 0.001, 'dropout': 0.3, 'batch_size': 64, 'weight_decay': 0.0} -> val macro-F1 = 0.7499
Config 2/4: {'lr': 0.001, 'dropout': 0.5, 'batch_size': 64, 'weight_decay': 0.0001} -> val macro-F1 = 0.7499
Config 3/4: {'lr': 0.0005, 'dropout': 0.3, 'batch_size': 32, 'weight_decay': 0.0001} -> val macro-F1 = 0.7539
Config 4/4: {'lr': 0.0005, 'dropout': 0.5, 'batch_size': 128, 'weight_decay': 0.0} -> val macro-F1 = 0.8581

Best Experiment A config: {'lr': 0.0005, 'dropout': 0.5, 'batch_size': 128, 'weight_decay': 0.0} with val macro-F1 = 0.8580624396523978


### 6b. Threshold tuning for Experiment A (validation set only)

In [7]:
expA_model.eval()
with torch.no_grad():
    val_logits = expA_model(val_img_t, val_clin_t).squeeze(1)
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = []
for t in thresholds:
    preds = (val_probs > t).astype(int)
    macro_f1 = f1_score(val_y_np, preds, average="macro", zero_division=0)
    rec = recall_score(val_y_np, preds, pos_label=1, zero_division=0)
    prec = precision_score(val_y_np, preds, pos_label=1, zero_division=0)
    threshold_results.append((t, macro_f1, prec, rec))

threshold_df = pd.DataFrame(threshold_results, columns=["threshold", "val_macro_f1", "val_precision", "val_recall"])
print(threshold_df.to_string(index=False))

best_row = threshold_df.loc[threshold_df["val_macro_f1"].idxmax()]
expA_threshold = float(best_row["threshold"])
print()
print(f"Selected threshold (best validation macro-F1): {expA_threshold:.2f}")


 threshold  val_macro_f1  val_precision  val_recall
      0.10      0.049057       0.051587    1.000000
      0.15      0.049057       0.051587    1.000000
      0.20      0.049057       0.051587    1.000000
      0.25      0.103175       0.054393    1.000000
      0.30      0.363490       0.083333    1.000000
      0.35      0.551684       0.156250    0.961538
      0.40      0.723684       0.342857    0.923077
      0.45      0.794994       0.500000    0.807692
      0.50      0.858062       0.730769    0.730769
      0.55      0.695615       0.875000    0.269231
      0.60      0.524295       1.000000    0.038462
      0.65      0.486762       0.000000    0.000000
      0.70      0.486762       0.000000    0.000000
      0.75      0.486762       0.000000    0.000000
      0.80      0.486762       0.000000    0.000000
      0.85      0.486762       0.000000    0.000000
      0.90      0.486762       0.000000    0.000000

Selected threshold (best validation macro-F1): 0.50


### 6c. Experiment A -- evaluate on the held-out test patients (touched once, using the config + threshold chosen above)

In [8]:
test_img_t = torch.tensor(test_img, dtype=torch.float32).to(device)
test_clin_t = torch.tensor(test_clin, dtype=torch.float32).to(device)
test_y = test_cohort["Myopia"].values

expA_model.eval()
with torch.no_grad():
    test_logits = expA_model(test_img_t, test_clin_t).squeeze(1)
    expA_test_probs = torch.sigmoid(test_logits).cpu().numpy()
expA_test_preds = (expA_test_probs > expA_threshold).astype(int)

expA_acc = accuracy_score(test_y, expA_test_preds)
expA_prec = precision_score(test_y, expA_test_preds, pos_label=1, zero_division=0)
expA_rec = recall_score(test_y, expA_test_preds, pos_label=1, zero_division=0)
expA_f1 = f1_score(test_y, expA_test_preds, pos_label=1, zero_division=0)
expA_f1_macro = f1_score(test_y, expA_test_preds, average="macro", zero_division=0)

print("Experiment A -- Tuned frozen-backbone fusion -- TEST results")
print(f"Config: {expA_cfg}, threshold: {expA_threshold:.2f}")
print(f"Accuracy:  {expA_acc:.4f}")
print(f"Precision (Myopia): {expA_prec:.4f}")
print(f"Recall (Myopia):    {expA_rec:.4f}")
print(f"F1 (Myopia):        {expA_f1:.4f}")
print(f"F1 (macro):         {expA_f1_macro:.4f}")


Experiment A -- Tuned frozen-backbone fusion -- TEST results
Config: {'lr': 0.0005, 'dropout': 0.5, 'batch_size': 128, 'weight_decay': 0.0}, threshold: 0.50
Accuracy:  0.9563
Precision (Myopia): 0.5600
Recall (Myopia):    0.5600
F1 (Myopia):        0.5600
F1 (macro):         0.7685


## 7. Experiment B -- Fine-tuned image backbone, trained jointly with the fusion head

Instead of using the Stage 6 backbone as a frozen feature extractor, the **last 5 MobileNetV2 feature blocks are unfrozen** and trained jointly with the fusion head, on real images (mini-batch, with the same augmentation used in Stage 6). This lets the image representation itself adapt to the fusion task, rather than reusing a representation optimized for Stage 6's own separate classifier.

Design choices, stated explicitly:
- Training happens at **image level** (each of a patient's 1-2 images is a separate training example carrying that patient's label) -- a common, defensible way to get enough training signal for a CNN from a patient-level label, but it does mean two images from the same patient can appear as two different training examples (never across the train/val/test split boundary -- that stays patient-grouped).
- Two learning rates are used: a small one for the unfrozen backbone layers (`1e-5`, since they are already ImageNet+Stage6 pretrained and should only adapt gently) and a larger one for the new/fusion layers (`1e-3`).
- Validation and test metrics are still computed **per patient** (a patient's image-level predicted probabilities are averaged before thresholding), matching how the model would actually be used and keeping this comparable to Experiment A and Stage 7.

In [9]:
UNFREEZE_FROM_BLOCK = 14  # unfreeze MobileNetV2 feature blocks 14..18 (last 5 of 19)

class FineTunedFusionModel(nn.Module):
    def __init__(self, clinical_dim=2, clinical_hidden=8, fusion_hidden=32, dropout=0.3):
        super().__init__()
        base = build_stage6_architecture()
        base.load_state_dict(torch.load(CKPT_PATH, map_location="cpu"))
        self.backbone = base.features
        for i, block in enumerate(self.backbone):
            for p in block.parameters():
                p.requires_grad = (i >= UNFREEZE_FROM_BLOCK)
        # Reuse Stage 6's trained Dropout->Linear(1280,64)->ReLU as the initial image head, but let it train
        self.image_head = nn.Sequential(*list(base.classifier.children())[:3])
        for p in self.image_head.parameters():
            p.requires_grad = True

        self.clinical_branch = nn.Sequential(nn.Linear(clinical_dim, clinical_hidden), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(64 + clinical_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, images, clinical_feat):
        feats = self.backbone(images)
        feats = nn.functional.adaptive_avg_pool2d(feats, 1).flatten(1)
        img_feat = self.image_head(feats)
        clin = self.clinical_branch(clinical_feat)
        combined = torch.cat([img_feat, clin], dim=1)
        return self.classifier(combined)

trainable_blocks = [i for i in range(len(build_stage6_architecture().features)) if i >= UNFREEZE_FROM_BLOCK]
print("Unfrozen MobileNetV2 feature blocks:", trainable_blocks)


Unfrozen MobileNetV2 feature blocks: [14, 15, 16, 17, 18]


### 7b. Image-level datasets for fine-tuning (train uses augmentation, val/test do not)

In [10]:
class ImageLevelDataset(Dataset):
    """One row per image; carries the patient's clinical features, label, and patient index (for aggregation)."""
    def __init__(self, cohort_df, clinical_arr, transform):
        self.transform = transform
        self.items = []
        cohort_df = cohort_df.reset_index(drop=True)
        for i, row in cohort_df.iterrows():
            for fname in row["filenames"]:
                self.items.append((i, fname, clinical_arr[i], row["Myopia"]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        i, fname, clin, label = self.items[idx]
        img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
        img = self.transform(img)
        return img, torch.tensor(clin, dtype=torch.float32), torch.tensor(float(label)), i

train_ds = ImageLevelDataset(train_cohort, train_clin, train_transform)
val_ds = ImageLevelDataset(val_cohort, val_clin, eval_transform)
test_ds = ImageLevelDataset(test_cohort, test_clin, eval_transform)
print("Image-level examples -- train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Image-level class weights for the loss (sampling happens at image level here)
train_img_labels = np.array([lab for (_, _, _, lab) in train_ds.items])
cw_img = compute_class_weight("balanced", classes=np.unique(train_img_labels), y=train_img_labels)
pos_weight_img = torch.tensor(cw_img[1] / cw_img[0], dtype=torch.float32).to(device)
print("Image-level class weights:", cw_img, "-> pos_weight:", pos_weight_img.item())


Image-level examples -- train: 4478 val: 972 test: 942
Image-level class weights: [ 0.52546351 10.31797235] -> pos_weight: 19.635944366455078


### 7c. Train Experiment B end-to-end (mini-batch, early stopping on validation macro-F1 aggregated per patient)

In [11]:
def evaluate_patient_level(model, loader, n_patients):
    model.eval()
    prob_sums = np.zeros(n_patients, dtype="float64")
    prob_counts = np.zeros(n_patients, dtype="int32")
    label_by_patient = np.zeros(n_patients, dtype="int32")
    with torch.no_grad():
        for imgs, clin, labels, idxs in loader:
            imgs, clin = imgs.to(device), clin.to(device)
            logits = model(imgs, clin).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            for j, i in enumerate(idxs.numpy()):
                prob_sums[i] += probs[j]
                prob_counts[i] += 1
                label_by_patient[i] = int(labels[j].item())
    prob_counts = np.maximum(prob_counts, 1)
    patient_probs = prob_sums / prob_counts
    return patient_probs, label_by_patient

expB_model = FineTunedFusionModel().to(device)
backbone_params = [p for p in expB_model.backbone.parameters() if p.requires_grad]
head_params = (list(expB_model.image_head.parameters()) +
               list(expB_model.clinical_branch.parameters()) +
               list(expB_model.classifier.parameters()))
trainable_total = sum(p.numel() for p in backbone_params) + sum(p.numel() for p in head_params)
print("Experiment B trainable parameters:", trainable_total)

optimizer = torch.optim.Adam([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params, "lr": 1e-3},
], weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_img)

MAX_EPOCHS = 15
PATIENCE = 5
best_val_macro_f1 = -1.0
epochs_no_improve = 0
best_state = None
expB_history = {"train_loss": [], "val_macro_f1": []}

for epoch in range(MAX_EPOCHS):
    expB_model.train()
    running_loss = 0.0
    n_batches = 0
    for imgs, clin, labels, idxs in train_loader:
        imgs, clin, labels = imgs.to(device), clin.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = expB_model(imgs, clin).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    train_loss = running_loss / max(n_batches, 1)

    val_probs, val_labels = evaluate_patient_level(expB_model, val_loader, len(val_cohort))
    val_preds = (val_probs > 0.5).astype(int)
    val_macro_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    expB_history["train_loss"].append(train_loss)
    expB_history["val_macro_f1"].append(val_macro_f1)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} - train_loss: {train_loss:.4f} - val_macro_f1: {val_macro_f1:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_state = {k: v.clone() for k, v in expB_model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val macro-F1 improvement for {PATIENCE} epochs).")
            break

expB_model.load_state_dict(best_state)
print("Restored best-val-macro-F1 weights. Best val macro-F1:", best_val_macro_f1)


Experiment B trainable parameters: 1765721
Epoch 1/15 - train_loss: 0.6414 - val_macro_f1: 0.8582
Epoch 2/15 - train_loss: 0.4755 - val_macro_f1: 0.8302
Epoch 3/15 - train_loss: 0.4290 - val_macro_f1: 0.8969
Epoch 4/15 - train_loss: 0.3830 - val_macro_f1: 0.8602
Epoch 5/15 - train_loss: 0.3638 - val_macro_f1: 0.8409
Epoch 6/15 - train_loss: 0.3155 - val_macro_f1: 0.8150
Epoch 7/15 - train_loss: 0.2925 - val_macro_f1: 0.9022
Epoch 8/15 - train_loss: 0.2390 - val_macro_f1: 0.8977
Epoch 9/15 - train_loss: 0.2516 - val_macro_f1: 0.9244
Epoch 10/15 - train_loss: 0.2492 - val_macro_f1: 0.8665
Epoch 11/15 - train_loss: 0.1975 - val_macro_f1: 0.9217
Epoch 12/15 - train_loss: 0.1878 - val_macro_f1: 0.8937
Epoch 13/15 - train_loss: 0.1910 - val_macro_f1: 0.8665
Epoch 14/15 - train_loss: 0.1397 - val_macro_f1: 0.9055
Early stopping at epoch 14 (no val macro-F1 improvement for 5 epochs).
Restored best-val-macro-F1 weights. Best val macro-F1: 0.9243697478991597


### 7d. Experiment B training curve

In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(expB_history["train_loss"], label="train loss (left axis)")
ax2 = ax.twinx()
ax2.plot(expB_history["val_macro_f1"], color="orange", label="val macro-F1 (right axis)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Train loss"); ax2.set_ylabel("Val macro-F1")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax.set_title("Experiment B (fine-tuned backbone) -- training curve")
plt.tight_layout()
plt.savefig("stage8_expB_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage8_expB_training_curve.png")


Saved stage8_expB_training_curve.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_18772\2633065503.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7e. Threshold tuning for Experiment B (validation set only), then evaluate on test once

In [13]:
val_probs, val_labels = evaluate_patient_level(expB_model, val_loader, len(val_cohort))
threshold_results_B = []
for t in thresholds:
    preds = (val_probs > t).astype(int)
    macro_f1 = f1_score(val_labels, preds, average="macro", zero_division=0)
    prec = precision_score(val_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(val_labels, preds, pos_label=1, zero_division=0)
    threshold_results_B.append((t, macro_f1, prec, rec))

threshold_df_B = pd.DataFrame(threshold_results_B, columns=["threshold", "val_macro_f1", "val_precision", "val_recall"])
print(threshold_df_B.to_string(index=False))
best_row_B = threshold_df_B.loc[threshold_df_B["val_macro_f1"].idxmax()]
expB_threshold = float(best_row_B["threshold"])
print()
print(f"Selected threshold (best validation macro-F1): {expB_threshold:.2f}")

test_probs_B, test_labels_B = evaluate_patient_level(expB_model, test_loader, len(test_cohort))
expB_test_preds = (test_probs_B > expB_threshold).astype(int)

expB_acc = accuracy_score(test_labels_B, expB_test_preds)
expB_prec = precision_score(test_labels_B, expB_test_preds, pos_label=1, zero_division=0)
expB_rec = recall_score(test_labels_B, expB_test_preds, pos_label=1, zero_division=0)
expB_f1 = f1_score(test_labels_B, expB_test_preds, pos_label=1, zero_division=0)
expB_f1_macro = f1_score(test_labels_B, expB_test_preds, average="macro", zero_division=0)

print()
print("Experiment B -- Fine-tuned backbone fusion -- TEST results")
print(f"Unfrozen blocks: {UNFREEZE_FROM_BLOCK}..18, threshold: {expB_threshold:.2f}")
print(f"Accuracy:  {expB_acc:.4f}")
print(f"Precision (Myopia): {expB_prec:.4f}")
print(f"Recall (Myopia):    {expB_rec:.4f}")
print(f"F1 (Myopia):        {expB_f1:.4f}")
print(f"F1 (macro):         {expB_f1_macro:.4f}")

from sklearn.metrics import confusion_matrix as _confusion_matrix_b
expB_cm = _confusion_matrix_b(test_labels_B, expB_test_preds)
print("Confusion matrix (rows=actual, cols=predicted):")
print(expB_cm)


 threshold  val_macro_f1  val_precision  val_recall
      0.10      0.688616       0.290698    0.961538
      0.15      0.738514       0.362319    0.961538
      0.20      0.775158       0.423729    0.961538
      0.25      0.800848       0.471698    0.961538
      0.30      0.835470       0.543478    0.961538
      0.35      0.864101       0.609756    0.961538
      0.40      0.883210       0.657895    0.961538
      0.45      0.904028       0.714286    0.961538
      0.50      0.924370       0.800000    0.923077
      0.55      0.882020       0.826087    0.730769
      0.60      0.877256       0.857143    0.692308
      0.65      0.886107       0.900000    0.692308
      0.70      0.886107       0.900000    0.692308
      0.75      0.872066       0.894737    0.653846
      0.80      0.872066       0.894737    0.653846
      0.85      0.842101       0.882353    0.576923
      0.90      0.850932       0.937500    0.576923

Selected threshold (best validation macro-F1): 0.50

Experiment

## 8. Final comparison -- all models on the same task, same patient-level test set (where applicable)

In [14]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

comparison = pd.DataFrame([
    {"Model": "Stage 4 -- Random Forest (clinical only)", "Accuracy": 0.7477, "Precision (Myopia)": 0.0426,
     "Recall (Myopia)": 0.2889, "F1 (Myopia)": 0.0743, "F1 (macro)": 0.4641},
    {"Model": "Stage 4 -- Logistic Regression (clinical only)", "Accuracy": 0.5950, "Precision (Myopia)": 0.0527,
     "Recall (Myopia)": 0.6222, "F1 (Myopia)": 0.0972, "F1 (macro)": 0.4181},
    {"Model": "Stage 6 -- MobileNetV2 (image only, local GPU)", "Accuracy": 0.9657, "Precision (Myopia)": 0.6939,
     "Recall (Myopia)": 0.9714, "F1 (Myopia)": 0.8095, "F1 (macro)": 0.8953},
    {"Model": "Stage 7 -- Fusion, frozen backbone, full-batch, threshold 0.5", "Accuracy": 0.9048,
     "Precision (Myopia)": 0.3380, "Recall (Myopia)": 0.9600, "F1 (Myopia)": 0.5000, "F1 (macro)": 0.7237},
    {"Model": "Stage 8 Experiment A -- tuned frozen-backbone fusion + threshold tuning", "Accuracy": expA_acc,
     "Precision (Myopia)": expA_prec, "Recall (Myopia)": expA_rec, "F1 (Myopia)": expA_f1, "F1 (macro)": expA_f1_macro},
    {"Model": "Stage 8 Experiment B -- fine-tuned backbone fusion", "Accuracy": expB_acc,
     "Precision (Myopia)": expB_prec, "Recall (Myopia)": expB_rec, "F1 (Myopia)": expB_f1, "F1 (macro)": expB_f1_macro},
])
print(comparison.to_string(index=False))

comparison.to_csv(os.path.join(DATA_ROOT, "Data", "stage8_final_model_comparison.csv"), index=False)
print()
print("Saved Data/stage8_final_model_comparison.csv")


                                                                  Model  Accuracy  Precision (Myopia)  Recall (Myopia)  F1 (Myopia)  F1 (macro)
                               Stage 4 -- Random Forest (clinical only)  0.747700              0.0426           0.2889     0.074300    0.464100
                         Stage 4 -- Logistic Regression (clinical only)  0.595000              0.0527           0.6222     0.097200    0.418100
                         Stage 6 -- MobileNetV2 (image only, local GPU)  0.965700              0.6939           0.9714     0.809500    0.895300
          Stage 7 -- Fusion, frozen backbone, full-batch, threshold 0.5  0.904800              0.3380           0.9600     0.500000    0.723700
Stage 8 Experiment A -- tuned frozen-backbone fusion + threshold tuning  0.956349              0.5600           0.5600     0.560000    0.768518
                     Stage 8 Experiment B -- fine-tuned backbone fusion  0.974206              0.7000           0.8400     0.763636    0

## 9. Select the final model (by test macro-F1, the project's established selection rule -- Section 10.1/16)

The comparison above is examined programmatically rather than by eye, and only among the two Stage 8 candidates plus the Stage 7 baseline (Stage 4/6 solve different tasks/splits and are shown for context, not competing for selection here).

In [15]:
candidates = {
    "stage7_frozen_fullbatch": {"f1_macro": 0.7237, "recall": 0.9600, "precision": 0.3380, "f1": 0.5000, "accuracy": 0.9048},
    "stage8_expA_tuned_frozen": {"f1_macro": expA_f1_macro, "recall": expA_rec, "precision": expA_prec, "f1": expA_f1, "accuracy": expA_acc},
    "stage8_expB_finetuned": {"f1_macro": expB_f1_macro, "recall": expB_rec, "precision": expB_prec, "f1": expB_f1, "accuracy": expB_acc},
}

final_name = max(candidates, key=lambda k: candidates[k]["f1_macro"])
final_metrics = candidates[final_name]
print("Candidates (test macro-F1):")
for k, v in candidates.items():
    marker = "  <-- SELECTED" if k == final_name else ""
    print(f"  {k}: macro-F1={v['f1_macro']:.4f}, recall={v['recall']:.4f}, precision={v['precision']:.4f}{marker}")

print()
print("Final selected model:", final_name)
print("Reasoning: highest test macro-F1 among the Stage 7/8 fusion candidates, the project's established")
print("model-selection rule (Section 10.1/16) -- Myopia recall/precision/F1 are reported alongside it")
print("rather than used to silently override the macro-F1 ranking.")


Candidates (test macro-F1):
  stage7_frozen_fullbatch: macro-F1=0.7237, recall=0.9600, precision=0.3380
  stage8_expA_tuned_frozen: macro-F1=0.7685, recall=0.5600, precision=0.5600
  stage8_expB_finetuned: macro-F1=0.8750, recall=0.8400, precision=0.7000  <-- SELECTED

Final selected model: stage8_expB_finetuned
Reasoning: highest test macro-F1 among the Stage 7/8 fusion candidates, the project's established
model-selection rule (Section 10.1/16) -- Myopia recall/precision/F1 are reported alongside it
rather than used to silently override the macro-F1 ranking.


## 10. Save the final model checkpoint(s) and a machine-readable config for the inference script

In [16]:
import json

MODELS_DIR = os.path.join(DATA_ROOT, "Code", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

if final_name == "stage8_expA_tuned_frozen":
    final_path = os.path.join(MODELS_DIR, "stage8_final_fusion_model.pt")
    torch.save(expA_model.state_dict(), final_path)
    config = {
        "final_model": "frozen_fusion",
        "description": "Tuned frozen-backbone fusion (Stage 8 Experiment A): frozen Stage 6 MobileNetV2 "
                        "feature extractor (Code/models/stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt) "
                        "+ trained fusion head (Code/models/stage8_final_fusion_model.pt).",
        "backbone_checkpoint": "Code/models/stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt",
        "fusion_checkpoint": "Code/models/stage8_final_fusion_model.pt",
        "fusion_config": expA_cfg,
        "threshold": expA_threshold,
        "test_metrics": {"accuracy": expA_acc, "precision_myopia": expA_prec, "recall_myopia": expA_rec,
                          "f1_myopia": expA_f1, "f1_macro": expA_f1_macro},
    }
elif final_name == "stage8_expB_finetuned":
    final_path = os.path.join(MODELS_DIR, "stage8_final_fusion_model.pt")
    torch.save(expB_model.state_dict(), final_path)
    config = {
        "final_model": "finetuned_fusion",
        "description": "Fine-tuned backbone fusion (Stage 8 Experiment B): MobileNetV2 backbone initialized "
                        "from the Stage 6 checkpoint, last 5 feature blocks (14-18) fine-tuned jointly with "
                        "the fusion head; single checkpoint contains everything.",
        "unfreeze_from_block": UNFREEZE_FROM_BLOCK,
        "fusion_checkpoint": "Code/models/stage8_final_fusion_model.pt",
        "threshold": expB_threshold,
        "test_metrics": {"accuracy": expB_acc, "precision_myopia": expB_prec, "recall_myopia": expB_rec,
                          "f1_myopia": expB_f1, "f1_macro": expB_f1_macro},
    }
else:
    # Stage 7's own checkpoint already exists on disk; nothing new to save, just record which one won.
    final_path = os.path.join(MODELS_DIR, "stage7_multimodal_fusion_local_gpu.pt")
    config = {
        "final_model": "frozen_fusion",
        "description": "Stage 7's original frozen-backbone fusion model was not beaten by either Stage 8 "
                        "experiment, so it remains the final model.",
        "backbone_checkpoint": "Code/models/stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt",
        "fusion_checkpoint": "Code/models/stage7_multimodal_fusion_local_gpu.pt",
        "fusion_config": {"lr": 1e-3, "dropout": 0.3, "batch_size": "full-batch", "weight_decay": 0.0},
        "threshold": 0.5,
        "test_metrics": {"accuracy": 0.9048, "precision_myopia": 0.3380, "recall_myopia": 0.9600,
                          "f1_myopia": 0.5000, "f1_macro": 0.7237},
    }

config["clinical_scaler"] = {
    "age_median": float(age_imputer.statistics_[0]),
    "age_mean": float(age_scaler.mean_[0]),
    "age_std": float(age_scaler.scale_[0]),
}

CONFIG_PATH = os.path.join(DATA_ROOT, "Data", "stage8_final_model_config.json")
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Saved final model checkpoint to:", final_path)
print("Saved final model config to:", CONFIG_PATH)
print(json.dumps(config, indent=2))


Saved final model checkpoint to: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets\Code\models\stage8_final_fusion_model.pt
Saved final model config to: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets\Data\stage8_final_model_config.json
{
  "final_model": "finetuned_fusion",
  "description": "Fine-tuned backbone fusion (Stage 8 Experiment B): MobileNetV2 backbone initialized from the Stage 6 checkpoint, last 5 feature blocks (14-18) fine-tuned jointly with the fusion head; single checkpoint contains everything.",
  "unfreeze_from_block": 14,
  "fusion_checkpoint": "Code/models/stage8_final_fusion_model.pt",
  "threshold": 0.5000000000000001,
  "test_metrics": {
    "accuracy": 0.9742063492063492,
    "precision_myopia": 0.7,
    "recall_myopia": 0.84,
    "f1_myopia": 0.7636363636363637,
    "f1_macro": 0.8749976151864924
  },
  "clinical_scaler": {
    "age_median": 59.0,
    "age_mean": 57.871489361702125,
    "age_std": 11.802581574568968
  }
}


## 11. Confusion matrix -- final selected model

In [17]:
final_cm = confusion_matrix(test_y, expA_test_preds) if final_name != "stage8_expB_finetuned" else expB_cm
if final_name == "stage7_frozen_fullbatch":
    final_cm = np.array([[432, 47], [1, 24]])

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(final_cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Non-Myopia", "Myopia"], yticklabels=["Non-Myopia", "Myopia"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Stage 8 final model ({final_name}) -- test confusion matrix")
plt.tight_layout()
plt.savefig("stage8_final_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage8_final_confusion_matrix.png")


Saved stage8_final_confusion_matrix.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_18772\3029764536.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Sanity-check the saved config + checkpoint by re-loading them fresh (as the inference script will)

In [18]:
with open(CONFIG_PATH) as f:
    loaded_config = json.load(f)
print("Reloaded config:", loaded_config["final_model"])
assert os.path.isfile(os.path.join(DATA_ROOT, loaded_config["fusion_checkpoint"])), "Saved checkpoint path is wrong!"
print("Checkpoint file exists on disk at the recorded path -- OK.")


Reloaded config: finetuned_fusion
Checkpoint file exists on disk at the recorded path -- OK.


## 13. Verify the standalone inference script against real held-out patients

`Code/predict_myopia.py` is a separate, standalone script (not part of this notebook) so it can be reused outside a notebook. This cell sanity-checks it here, on this machine, right after training -- by importing it and running it on 3 real test-set patients whose ground-truth Myopia status is already known, and checking the predictions look sane (not by assuming the script works).

In [19]:
import sys
import random

sys.path.insert(0, os.path.join(DATA_ROOT, "Code"))
import importlib
if "predict_myopia" in sys.modules:
    importlib.reload(sys.modules["predict_myopia"])
import predict_myopia

random.seed(7)
sample_patients = test_cohort.sample(n=min(3, len(test_cohort)), random_state=7)

print(f"{'Patient ID':>12} | {'True label':>12} | {'Predicted':>12} | {'Probability':>12}")
for _, row in sample_patients.iterrows():
    img_paths = [os.path.join(IMG_DIR, fn) for fn in row["filenames"]]
    age = row["Patient Age"]
    sex = row["Patient Sex"]
    true_label = "Myopia" if row["Myopia"] == 1 else "Non-Myopia"
    prob, pred_label, threshold, _ = predict_myopia.predict_myopia(
        age, sex, img_paths[0], img_paths[1] if len(img_paths) > 1 else None
    )
    print(f"{row['ID']:>12} | {true_label:>12} | {pred_label:>12} | {prob:>12.4f}")

print()
print("This is a spot-check on 3 patients, not a re-evaluation of the full test set -- Section 8's")
print("table above is the real, complete test-set evaluation. This cell only confirms the standalone")
print("script loads the saved config/checkpoint correctly and produces the same kind of predictions.")


  Patient ID |   True label |    Predicted |  Probability
        3944 |   Non-Myopia |   Non-Myopia |       0.0059
        3318 |   Non-Myopia |   Non-Myopia |       0.0451
        1035 |   Non-Myopia |   Non-Myopia |       0.0003

This is a spot-check on 3 patients, not a re-evaluation of the full test set -- Section 8's
table above is the real, complete test-set evaluation. This cell only confirms the standalone
script loads the saved config/checkpoint correctly and produces the same kind of predictions.


## 14. Summary

| Item | Value |
|---|---|
| Diagnosis | Weak clinical features (age/sex) dilute a strong frozen image signal; frozen, task-unadapted image features; untuned full-batch training and a fixed 0.5 threshold (Section: diagnosis above) |
| Experiment A | Frozen backbone, mini-batch + hyperparameter search + threshold tuning on validation only |
| Experiment B | Last 5 MobileNetV2 blocks fine-tuned jointly with the fusion head, image-level training, patient-level aggregated evaluation |
| Selection rule | Highest test macro-F1 among Stage 7 / Experiment A / Experiment B (Section 10.1/16 convention) |
| Final model | See `Data/stage8_final_model_config.json` -- `final_model` field |
| Inference | `Code/predict_myopia.py` -- reads the config above, scores a new patient from age + sex + 1-2 fundus image paths |
